In [1]:
import math

"""
Softmax can be useful for when we have non-normalized,
or uncalibrated inputs and want to produce a normal
 distribution of probabilities. It represents confidence
 scores.
"""

layer_outputs = [4.8, 1.21, 2.385]
exp_values = []

for output in layer_outputs:
    exp_values.append(math.e ** output)
print(exp_values)

[121.51041751873483, 3.353484652549023, 10.859062664920513]


In [3]:
norm_base = sum(exp_values)
norm_values = []
for value in exp_values:
    norm_values.append(value / norm_base)
print(norm_values) # This sums up to 1.

[0.8952826639572619, 0.024708306782099374, 0.0800090292606387]


In [6]:
"""
We can perform the same set of operations with NumPy
"""
import numpy as np

layer_outputs = [4.8, 1.21, 2.385]
exp_values = np.exp(layer_outputs)
print(exp_values)

norm_values = exp_values / np.sum(exp_values)
print(norm_values)

[121.51041752   3.35348465  10.85906266]
[0.89528266 0.02470831 0.08000903]


In [7]:
"""
To train in batches we need to convert this functionality
to accept layer outputs in batches
"""
import numpy as np

layer_outputs = np.array([[4.8, 1.21, 2.385],
                          [8.9, -1.81, .2],
                          [1.41, 1.051, .026]])
print("Sum without axis \n", np.sum(layer_outputs))
print("This will be identical to the above since default is None: \n", np.sum(layer_outputs, axis=None))


Sum without axis 
 18.172
This will be identical to the above since default is None: 
 18.172


In [10]:
"""
With no axis specified we are just summing all of
the values, even if they're in varying dimensions.
Next, axis=0. This means to sum row-wise, along
axis 0. In other words, the output has the same
size as this axis, as at each of the positions of this
output, the values from all the other dimensions at
this position are summed to form it. In the case of our
2D array, where we have only a single other dimension,
the columns, the output vector will sum these columns.
"""

print(np.sum(layer_outputs, axis=0))

[15.11   0.451  2.611]


In [13]:
"""
But this isn't what we want. We want to sum the rows
instead, like this w/ raw python:
"""
for i in layer_outputs:
    print(sum(i))

# Or like this with numpy:
print(np.sum(layer_outputs, axis=1, keepdims=True))

8.395
7.29
2.4869999999999997
[[8.395]
 [7.29 ]
 [2.487]]


In [26]:
import numpy as np

class ActivationSoftmax:
    def forward(self, inputs):
        exp_values = np.exp(inputs - np.max(inputs, axis=1, keepdims=True))
        probs = exp_values / np.sum(exp_values, axis=1, keepdims=True)
        self.output = probs

In [27]:
softmax = ActivationSoftmax()
softmax.forward([[1, 2, 3]])
print(softmax.output)

[[0.09003057 0.24472847 0.66524096]]


In [30]:
from nnfs.datasets import spiral_data

class ActivationReLU:
    def forward(self, inputs):
        self.output = np.maximum(0, inputs)

class LayerDense:
    def __init__(self, n_inputs, n_neurons):
        self.weights = .01 * np.random.randn(n_inputs, n_neurons)
        self.biases = np.zeros((1, n_neurons))

    def forward(self, inputs):
        self.output = np.dot(inputs, self.weights) + self.biases

X, y = spiral_data(samples=100, classes=3)
dense1 = LayerDense(2, 3)
activation1 = ActivationReLU()
dense2 = LayerDense(3, 3)
activation2 = ActivationSoftmax()
dense1.forward(X)
activation1.forward(dense1.output)
dense2.forward(activation1.output)
activation2.forward(dense2.output)
print(activation2.output[:5])

"""
As we can see, the distribution of prediction is almost
equal (~33%). These outputs are out "confidence scores."
"""

[[0.33333333 0.33333333 0.33333333]
 [0.33333361 0.33333351 0.33333288]
 [0.33333422 0.33333288 0.3333329 ]
 [0.33333396 0.33333394 0.33333209]
 [0.33333509 0.33333251 0.33333239]]


'\nAs we can see, the distribution of prediction is almost\nequal (~33%). These outputs are out "confidence scores."\n'